# 5단계: 묶음별 학습·평가 (robomimic BC / BC-RNN + MuJoCo 롤아웃)

- 입력: `train_evaluate(stage="prepare")`가 만든 학습 계획 (`plan_b64.txt` 한 줄)
- 출력: 묶음별 성공률 → 마지막 셀의 `RESULTS_JSON` 을 로컬 `runs/<run_id>/results/results.json` 으로 저장 → `train_evaluate(stage="collect")`
- 공정 비교: 모든 묶음이 같은 학습 설정·시드로 학습하고, **같은 50개 초기 상태(평가 시드 고정)** 에서 평가한다
- MODE = "pilot": train과 같은 개수 무작위 두 묶음만, BC와 BC-RNN을 짧게 학습해 체크포인트별 성공률과 시간을 잰다 (천장/바닥 효과 확인)
- MODE = "main": 계획의 모든 묶음을 계획 설정으로 학습·평가한다. 묶음 하나가 끝날 때마다 결과를 출력하므로 세션이 끊겨도 앞 결과는 남는다

In [ ]:
# 1. GPU와 파이썬 버전 확인
import sys, torch
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
print('[결과] python', sys.version.split()[0], '| torch', torch.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
# 2. 설치 (10/11 코랩에서 확인한 문제 3가지를 반영)
#  - 저장소를 /content 바로 아래에 받으면 코랩이 sys.path에 /content를 넣어 두어서, 설치된 패키지 대신 저장소 폴더를 import 한다 → /content/src 아래에 받는다
#  - robosuite 1.5.1은 mujoco>=3.2.3 만 요구해 최신 3.15가 깔리는데, 관절 처리에서 AssertionError가 난다 → mujoco 3.3.0으로 고정
#  - robomimic은 transformers==4.41.2 등을 고정 요구해 Python 3.13에서 빌드 실패 → 의존성 없이 설치하고 필요한 패키지만 따로 설치
#  - 편집 가능 설치(-e)는 세션 재시작 전까지 import가 안 되므로 일반 설치를 쓴다
import time; t0 = time.time()
%cd /content
!mkdir -p src
![ -d src/robosuite ] || git clone -q https://github.com/ARISE-Initiative/robosuite.git src/robosuite
!cd src/robosuite && git checkout -q v1.5.1
![ -d src/robomimic ] || git clone -q https://github.com/ARISE-Initiative/robomimic.git src/robomimic
!pip install -q ./src/robosuite
!pip install -q --no-deps ./src/robomimic
!pip install -q psutil tqdm termcolor tensorboard tensorboardX imageio imageio-ffmpeg matplotlib egl_probe huggingface_hub transformers diffusers
# robosuite 설치 과정(mink 의존성)에서 mujoco가 최신으로 올라가므로, 마지막에 3.3.0으로 강제로 되돌린다
!pip install -q --no-deps --force-reinstall "mujoco==3.3.0"
import robosuite, robomimic, mujoco
print('[결과] robosuite', robosuite.__version__, '| mujoco', mujoco.__version__, '| robomimic', robomimic.__file__)
print(f'[결과] 설치 {time.time()-t0:.0f}초')

In [ ]:
# 3. 데이터 다운로드 (Can MH low_dim, 107.6MB) 
import os
os.environ['MUJOCO_GL'] = 'egl'   # 화면 없는 서버용 렌더링 설정
os.makedirs('/content/data', exist_ok=True)
DATA = '/content/data/can_mh_low_dim.hdf5'
if not os.path.exists(DATA):
    !wget -q -O {DATA} 'https://huggingface.co/datasets/robomimic/robomimic_datasets/resolve/main/v1.5/can/mh/low_dim_v15.hdf5?download=true'
!ls -la {DATA}

In [ ]:
# 4. 학습 계획 붙여 넣기 (plan_b64.txt 내용 한 줄) + 모드 선택
MODE = 'pilot'   # 'pilot' 또는 'main'
PLAN_B64 = ''
import json, base64, zlib
PLAN = json.loads(zlib.decompress(base64.b64decode(PLAN_B64)))
SETS = {k: [f'demo_{i}' for i in v] for k, v in PLAN['sets'].items()}
print('[결과] run_id', PLAN['run_id'], '| 지문', PLAN['fp'], '| 설정', PLAN['config'])
print('[결과] 묶음', {k: len(v) for k, v in SETS.items()}, '| 순서', PLAN['order'])

In [ ]:
# 5. 묶음별 필터 키를 넣은 학습용 데이터 복사본 (원본은 그대로)
import shutil, h5py, numpy as np
PDATA = '/content/data/plan.hdf5'
shutil.copy(DATA, PDATA)
with h5py.File(PDATA, 'a') as f:
    for k, ids in SETS.items():
        key = f'mask/{k}'
        if key in f:
            del f[key]
        f.create_dataset(key, data=np.array(ids, dtype='S'))
    print('[결과] 필터 키', {k: len(f[f'mask/{k}']) for k in SETS})

In [ ]:
# 6. 학습·평가 함수
import contextlib, glob, os, time, torch, numpy as np
from robomimic.config import config_factory
from robomimic.scripts.train import train
import robomimic.utils.file_utils as FileUtils
DEV = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
EVAL_SEED = 12345  # 모든 묶음이 같은 초기 상태에서 평가받도록 고정

def make_config(cfg, set_name, out_dir, save_every):
    config = config_factory(algo_name='bc')
    with config.values_unlocked():
        config.experiment.name = f"{set_name}_{cfg['algo']}"
        config.train.data = [{'path': PDATA}]
        config.train.output_dir = out_dir
        config.train.hdf5_filter_key = set_name
        config.train.num_epochs = cfg['num_epochs']
        config.experiment.epoch_every_n_steps = cfg['steps_per_epoch']
        config.train.batch_size = cfg['batch_size']
        config.train.seed = cfg['seed']
        config.train.cuda = torch.cuda.is_available()
        config.experiment.validate = False
        config.experiment.rollout.enabled = False   # 평가는 학습 뒤 아래 evaluate()로 따로 한다
        config.experiment.render_video = False
        config.experiment.logging.log_tb = False
        config.experiment.save.enabled = True
        config.experiment.save.every_n_epochs = save_every
        config.experiment.save.on_best_validation = False
        config.experiment.save.on_best_rollout_return = False
        config.experiment.save.on_best_rollout_success_rate = False
        config.observation.modalities.obs.low_dim = ['robot0_eef_pos', 'robot0_eef_quat', 'robot0_gripper_qpos', 'object']
        if cfg['algo'] == 'bc_rnn':   # robomimic 논문의 low_dim BC-RNN 설정
            config.algo.rnn.enabled = True
            config.algo.rnn.horizon = 10
            config.algo.rnn.hidden_dim = 400
            config.algo.rnn.num_layers = 2
            config.train.seq_length = 10
            config.algo.gmm.enabled = True
            config.algo.actor_layer_dims = ()
    return config

def evaluate(ckpt, n, horizon):
    policy, ckpt_dict = FileUtils.policy_from_checkpoint(ckpt_path=ckpt, device=DEV, verbose=False)
    env, _ = FileUtils.env_from_checkpoint(ckpt_dict=ckpt_dict, render=False, render_offscreen=False, verbose=False)
    success = 0
    for i in range(n):
        np.random.seed(EVAL_SEED + i)   # i번째 평가는 모든 묶음에서 같은 캔 초기 위치
        policy.start_episode()
        obs = env.reset()
        for t in range(horizon):
            obs, r, done, _ = env.step(policy(ob=obs))
            if env.is_success()['task']:
                success += 1
                break
    return success

def run_set(set_name, cfg, eval_n):
    # eval_n: {평가할 epoch: 롤아웃 횟수}
    eval_epochs = sorted(eval_n)
    out_dir = f"/content/runs/{set_name}_{cfg['algo']}"
    shutil.rmtree(out_dir, ignore_errors=True)
    save_every = eval_epochs[0] if len(eval_epochs) > 1 else cfg['num_epochs']
    t0 = time.time()
    # 학습 로그는 파일로 보낸다 (코랩 출력 창이 길어지면 앞의 결과 줄이 밀려나기 때문)
    os.makedirs('/content/logs', exist_ok=True)
    with open(f"/content/logs/{set_name}_{cfg['algo']}.log", 'w') as lf, contextlib.redirect_stdout(lf), contextlib.redirect_stderr(lf):
        train(make_config(cfg, set_name, out_dir, save_every), device=DEV)
    train_sec = time.time() - t0
    rows = {}
    for ep in eval_epochs:
        ckpt = sorted(glob.glob(f'{out_dir}/**/models/model_epoch_{ep}.pth', recursive=True))[-1]
        t1 = time.time()
        rows[ep] = {'success': evaluate(ckpt, eval_n[ep], cfg['rollout_horizon']), 'rollouts': eval_n[ep], 'eval_sec': round(time.time() - t1)}
    return train_sec, rows
print('[결과] 함수 준비, 장치', DEV)

In [ ]:
# 7. 실행 (파일럿과 본 실험 결과는 다른 변수에 담는다)
import datetime
RESULTS = globals().setdefault('RESULTS_' + MODE.upper(), {})   # 세션이 살아 있으면 이어서 (끝난 묶음은 건너뜀)
if MODE == 'pilot':
    rnd = next(k for k in SETS if k.startswith('random_') and k != 'random_frames')
    for algo in ['bc', 'bc_rnn']:
        cfg = dict(PLAN['config'], algo=algo, num_epochs=400)
        for name in ['train', rnd]:
            key = f'{name}|{algo}'
            if key in RESULTS:
                continue
            train_sec, rows = run_set(name, cfg, {100: 10, 200: 10, 300: 10, 400: 10})
            RESULTS[key] = {'train_sec': round(train_sec), 'by_epoch': rows}
            print('RESULT', json.dumps({key: RESULTS[key]}))
else:
    cfg = PLAN['config']
    eval_n = {cfg['num_epochs']: cfg['rollout_n']}
    if cfg.get('early_eval_epoch'):
        eval_n[cfg['early_eval_epoch']] = cfg['early_eval_n']
    for name in PLAN['order']:
        if name in RESULTS:
            continue
        train_sec, rows = run_set(name, cfg, eval_n)
        fin = rows[cfg['num_epochs']]
        RESULTS[name] = {'success': fin['success'], 'rollouts': fin['rollouts'], 'train_sec': round(train_sec),
                         'eval_sec': sum(r['eval_sec'] for r in rows.values()), 'by_epoch': {str(k): v for k, v in rows.items()}}
        print('RESULT', json.dumps({name: RESULTS[name]}), datetime.datetime.now().strftime('%H:%M'))

In [ ]:
# 8. 결과 (이 셀 출력의 JSON을 로컬 results.json으로 저장)
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'
out = {'fingerprint': PLAN['fp'], 'run_id': PLAN['run_id'], 'mode': MODE, 'config': dict(PLAN['config'], n_seeds=1),
       'gpu': gpu, 'eval_seed': EVAL_SEED, 'finished_at': datetime.datetime.now().isoformat(timespec='seconds'), 'sets': RESULTS}
print('RESULTS_JSON_BEGIN'); print(json.dumps(out)); print('RESULTS_JSON_END')